# Blank-image control: is the image doing the work?

The fine-tuned adapter scores **51.5%** dim on 406 held-out depth/width/height rows, and
**70.8%** on the 257 rows with 2+ OCR candidates where one is correct. The best text-only
rule (fitted on training rows) scores **46.3%** on those 257; uniform random is 35.5%.

Option shuffling during training removed *position* as a shortcut, but not the candidate
text itself. This notebook runs the same adapter, same prompt, same processor, and changes
only the pixels:

| condition | image given | what it tests |
|---|---|---|
| `real` | the product photo | reproduces the training run - proves the adapter loaded |
| `blank` | white image, **same size** as the photo | removes all visual content, keeps the visual token count |
| `swapped` | another product's photo | a real photo, but the wrong one |

Reading it: if `blank` and `swapped` stay near 70.8% on the 2+ rows, the model learned a
text prior and the image is decoration. If they fall toward 46% or below while `real` holds,
the image is doing the work.


In [ ]:
import glob, os, subprocess, sys, json, math, random, re, string, time
from collections import Counter

MODEL_ID   = "Qwen/Qwen2-VL-2B-Instruct"
MAX_PIXELS = 512 * 512           # identical to training
SEED       = 7
CONDITIONS = ["real", "blank", "swapped"]   # real first: it validates the other two

found = glob.glob("/kaggle/input/**/eval.csv", recursive=True)
if not found:
    raise SystemExit("eval.csv not found - attach the fine-tuning-data dataset")
DEST = os.path.dirname(found[0])
cfg = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not cfg:
    raise SystemExit("adapter_config.json not found - attach the nlp-sem-ft notebook output")
ADAPTER = os.path.dirname(cfg[0])
prior = glob.glob("/kaggle/input/**/vlm_lora_results.json", recursive=True)
PRIOR = prior[0] if prior else None
WORK = "/kaggle/working"
print("data   :", DEST)
print("adapter:", ADAPTER, sorted(os.listdir(ADAPTER)))
print("prior  :", PRIOR)

for _k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF"):
    os.environ[_k] = "expandable_segments:True"
import torch
print("cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("No GPU")


In [ ]:
# Same repair as the training run: Kaggle's torchao 0.10.0 breaks transformers 5.x on import.
from importlib.metadata import PackageNotFoundError, version as _v
import packaging.version as V
def ver(p):
    try: return _v(p)
    except PackageNotFoundError: return None
tv, tao, pv = ver("transformers"), ver("torchao"), ver("peft")
print("before:  transformers", tv, "| torchao", tao, "| peft", pv)
if tao and tv and V.parse(tao) < V.parse("0.16.0") and V.parse(tv) >= V.parse("4.50"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"], check=False)
if pv is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--no-deps", "peft"], check=True)
print("after:   transformers", ver("transformers"), "| torchao", ver("torchao"), "| peft", ver("peft"))
import transformers, peft
print("imports OK")


In [ ]:
import csv
from PIL import Image
sys.path.insert(0, DEST)
import normalize as N

# Verbatim from the training notebook - the prompt must match character for character.
PHRASE = {"depth": "depth (front to back)",
          "width": "width (left to right)",
          "height": "height (top to bottom)"}
SELECT = ("This product photograph shows measurements. Text recognition read "
          "these values from the image:\n{options}\n"
          "Which one is the product's {phrase}?\n"
          "Answer with only the letter.")

def parse_cands(s):
    out = []
    for part in (s or "").split(";"):
        m = re.match(r"\s*(-?\d+(?:\.\d+)?)\s+(\S+)\s*$", part)
        if m:
            u = N.normalize_unit(m.group(2))
            if u:
                out.append((float(m.group(1)), u))
    return out

def render(row, order):
    opts = "\n".join(f"  ({string.ascii_lowercase[j]}) {row['cands'][k][0]:g} {row['cands'][k][1]}"
                      for j, k in enumerate(order))
    return SELECT.format(options=opts, phrase=PHRASE[row["attribute"]])

def dim_ok(pred, lo, hi, gu, tol=0.01):
    if pred is None: return False
    v, u = pred
    if N.family(u) != N.family(gu): return False
    a = N.to_base(v, u)
    return N.to_base(lo, gu)*(1-tol) <= a <= N.to_base(hi, gu)*(1+tol)

rows = []
for r in csv.DictReader(open(f"{DEST}/eval.csv")):
    r["cands"] = parse_cands(r["candidates"]); r["gold_index"] = int(r["gold_index"])
    rows.append(r)
multi = [i for i, r in enumerate(rows) if len(r["cands"]) >= 2]
print(f"eval rows {len(rows)} | re-asked under each condition: {len(multi)} (2+ candidates)")

# Swapped: a fixed derangement over the re-asked rows, never the row's own photo.
rng = random.Random(SEED)
files = [rows[i]["image_file"] for i in multi]
while True:
    perm = files[:]; rng.shuffle(perm)
    if all(a != b for a, b in zip(files, perm)):
        break
SWAP = dict(zip(multi, perm))

def image_for(i, condition):
    real = Image.open(f"{DEST}/images/{rows[i]['image_file']}").convert("RGB")
    if condition == "real":
        return real
    if condition == "blank":
        # Same dimensions as the photo, so the processor emits the same number of
        # visual tokens - only the content is removed.
        return Image.new("RGB", real.size, (255, 255, 255))
    if condition == "swapped":
        return Image.open(f"{DEST}/images/{SWAP[i]}").convert("RGB")
    raise ValueError(condition)


In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=MAX_PIXELS)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, attn_implementation="sdpa").to("cuda")
model = PeftModel.from_pretrained(base, ADAPTER).eval()
model.config.use_cache = True
n_lora = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
print(f"LoRA parameters loaded: {n_lora:,}  (training reported 18,464,768)")
assert n_lora == 18_464_768, "adapter did not load as trained"

def ask(img, prompt):
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
    chat = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = proc(text=[chat], images=[img], return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=6, do_sample=False)
    return proc.batch_decode(gen[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()


In [ ]:
results = {}
for cond in CONDITIONS:
    t0 = time.time(); out = []
    for step, i in enumerate(multi):
        r = rows[i]; cands = r["cands"]
        order = list(range(len(cands)))
        raw = ask(image_for(i, cond), render(r, order))
        m = re.search(r"\b([a-h])\b", raw.lower())
        j = string.ascii_lowercase.index(m.group(1)) if m else None
        pred = cands[order[j]] if (j is not None and j < len(cands)) else None
        out.append({"row": i, "image_file": r["image_file"], "attribute": r["attribute"],
                    "gold_index": r["gold_index"], "n_cands": len(cands),
                    "shown_image": SWAP[i] if cond == "swapped" else r["image_file"],
                    "raw": raw, "letter": m.group(1) if m else None,
                    "pred": list(pred) if pred else None,
                    "dim": bool(dim_ok(pred, float(r["gold_lo"]), float(r["gold_hi"]), r["gold_unit"]))})
        if (step+1) % 50 == 0:
            gp = [x for x in out if x["gold_index"] >= 0]
            print(f"  {cond:8s} {step+1}/{len(multi)}  2+ gold-present {100*sum(x['dim'] for x in gp)/max(1,len(gp)):.1f}%"
                  f"  {(time.time()-t0)/60:.1f} min", flush=True)
    results[cond] = out
    json.dump({"conditions": results, "swap": {str(k): v for k, v in SWAP.items()}},
              open(f"{WORK}/vlm_control_results.json", "w"), indent=1)
    print(f"{cond} done in {(time.time()-t0)/60:.1f} min - saved", flush=True)


In [ ]:
def wilson(k, n, z=1.96):
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return 100*(c-h)/d, 100*(c+h)/d

# Rows that do not depend on the image at all (0 or 1 candidate) - identical in every condition.
fixed = [r for i, r in enumerate(rows) if len(r["cands"]) < 2]
fixed_ok = sum(dim_ok(r["cands"][0] if r["cands"] else None, float(r["gold_lo"]), float(r["gold_hi"]), r["gold_unit"]) for r in fixed)

gold = Counter(string.ascii_lowercase[rows[i]["gold_index"]] for i in multi if rows[i]["gold_index"] >= 0)
print(f"{'condition':<10}{'all 406':>18}{'2+ gold-present':>28}{'  letters (a) (b) (c) (d) (e-h)':>34}")
print("-"*90)
for cond, out in results.items():
    k_all = fixed_ok + sum(x["dim"] for x in out)
    gp = [x for x in out if x["gold_index"] >= 0]; k = sum(x["dim"] for x in gp)
    lo, hi = wilson(k, len(gp))
    L = Counter(x["letter"] for x in gp if x["letter"]); t = sum(L.values()) or 1
    let = " ".join(f"{100*L[c]/t:4.1f}" for c in "abcd") + f" {100*sum(L[c] for c in 'efgh')/t:4.1f}"
    print(f"{cond:<10}{k_all:>5}/406 {100*k_all/406:5.1f}%   {k:>4}/{len(gp)} {100*k/len(gp):5.1f}% [{lo:.1f}, {hi:.1f}]   {let}")
t = sum(gold.values())
print(f"{'gold':<10}{'':>18}{'':>28}   " + " ".join(f"{100*gold[c]/t:4.1f}" for c in "abcd") + f" {100*sum(gold[c] for c in 'efgh')/t:4.1f}")
print("\nreference on the same 257 rows: best text-only rule 46.3% | uniform random 35.5% | zero-shot 43.3%")

# Did `real` reproduce the training run row for row?
if PRIOR:
    prev = {(x["image_file"], x["attribute"]): x for x in json.load(open(PRIOR))}
    same = sum(prev[(x["image_file"], x["attribute"])]["raw"] == x["raw"] for x in results["real"])
    print(f"\nreal vs training-run answers: {same}/{len(results['real'])} identical")

# How often does each control change the model's answer relative to real?
real = {x["row"]: x for x in results["real"]}
for cond in ("blank", "swapped"):
    ch = sum(real[x["row"]]["letter"] != x["letter"] for x in results[cond])
    gp = [x for x in results[cond] if x["gold_index"] >= 0]
    lost = sum(real[x["row"]]["dim"] and not x["dim"] for x in gp)
    gained = sum(x["dim"] and not real[x["row"]]["dim"] for x in gp)
    print(f"{cond:8s}: answer changed on {ch}/{len(results[cond])} rows | on gold-present rows lost {lost}, gained {gained}")
print("\nDownload vlm_control_results.json from Output.")
